<a href="https://colab.research.google.com/github/Samarthsaxena04/NLPASSG2-2305327-/blob/main/NLPAssg2(2305327).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets tokenizers evaluate torch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset

dataset = load_dataset("squad")
train_data = dataset["train"].select(range(1500))
val_data = dataset["validation"].select(range(500))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [ ]:
import os
from transformers import BertTokenizerFast, PreTrainedTokenizerFast
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace, CharDelimiterSplit

all_texts = []
for example in train_data:
    all_texts.append(example["context"])
    all_texts.append(example["question"])

wp_tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

bpe_tokenizer_raw = Tokenizer(BPE(unk_token="[UNK]"))
bpe_tokenizer_raw.pre_tokenizer = Whitespace()
bpe_trainer = BpeTrainer(
    vocab_size=30522,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"]
)
bpe_tokenizer_raw.train_from_iterator(all_texts, trainer=bpe_trainer)

os.makedirs("bpe_tokenizer", exist_ok=True)
bpe_tokenizer_raw.save("bpe_tokenizer/tokenizer.json")
bpe_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="bpe_tokenizer/tokenizer.json",
    unk_token="[UNK]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    pad_token="[PAD]",
    mask_token="[MASK]"
)

char_tokenizer_raw = Tokenizer(BPE(unk_token="[UNK]"))
char_tokenizer_raw.pre_tokenizer = CharDelimiterSplit(" ")
char_trainer = BpeTrainer(
    vocab_size=500,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"]
)
char_tokenizer_raw.train_from_iterator(all_texts, trainer=char_trainer)

os.makedirs("char_tokenizer", exist_ok=True)
char_tokenizer_raw.save("char_tokenizer/tokenizer.json")
char_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="char_tokenizer/tokenizer.json",
    unk_token="[UNK]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    pad_token="[PAD]",
    mask_token="[MASK]"
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
import torch
import time
import evaluate
from transformers import BertForQuestionAnswering, get_scheduler
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SQuADDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=384):
        self.samples = []
        for example in data:
            question = example["question"]
            context = example["context"]
            answer = example["answers"]["text"][0]
            ans_start = example["answers"]["answer_start"][0]

            encoding = tokenizer(
                question, context,
                max_length=max_length,
                truncation=True,
                padding="max_length",
                return_tensors="pt",
                return_offsets_mapping=True
            )

            offset_mapping = encoding["offset_mapping"][0]
            start_pos, end_pos = 0, 0
            ans_end = ans_start + len(answer)
            for idx, (s, e) in enumerate(offset_mapping.tolist()):
                if s <= ans_start < e:
                    start_pos = idx
                if s < ans_end <= e:
                    end_pos = idx
                    break

            self.samples.append({
                "input_ids": encoding["input_ids"][0],
                "attention_mask": encoding["attention_mask"][0],
                "start_positions": torch.tensor(start_pos),
                "end_positions": torch.tensor(end_pos)
            })

    def __len__(self): return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


def train_model(tokenizer, name):
    train_dataset = SQuADDataset(train_data, tokenizer)
    val_dataset = SQuADDataset(val_data, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=8)

    model = BertForQuestionAnswering.from_pretrained("bert-base-uncased")
    if tokenizer.vocab_size != 30522:
        model.resize_token_embeddings(len(tokenizer))
    model.to(device)

    optimizer = AdamW(model.parameters(), lr=2e-5)
    num_epochs = 2
    scheduler = get_scheduler(
        "linear", optimizer=optimizer,
        num_warmup_steps=50,
        num_training_steps=len(train_loader) * num_epochs
    )

    start_time = time.time()
    model.train()
    for epoch in range(num_epochs):
        for batch in train_loader:
            outputs = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                start_positions=batch["start_positions"].to(device),
                end_positions=batch["end_positions"].to(device)
            )
            outputs.loss.backward()
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
    training_time = time.time() - start_time

    model.eval()
    predictions, references = [], []
    with torch.no_grad():
        for i, batch in enumerate(val_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            for j in range(input_ids.size(0)):
                start_idx = outputs.start_logits[j].argmax().item()
                end_idx = outputs.end_logits[j].argmax().item()
                if end_idx < start_idx:
                    end_idx = start_idx

                tokens = input_ids[j][start_idx : end_idx + 1]
                pred_text = tokenizer.decode(tokens, skip_special_tokens=True)

                sample_idx = i * 8 + j
                if sample_idx < len(val_data):
                    q_id = val_data[sample_idx]["id"]
                    predictions.append({"id": q_id, "prediction_text": pred_text})
                    references.append({"id": q_id, "answers": val_data[sample_idx]["answers"]})

    squad_metric = evaluate.load("squad")
    scores = squad_metric.compute(predictions=predictions, references=references)
    param_count = sum(p.numel() for p in model.parameters())

    return {
        "name": name,
        "accuracy": scores["exact_match"],
        "f1": scores["f1"],
        "training_time": training_time,
        "parameters": param_count
    }


all_results = []
all_results.append(train_model(wp_tokenizer, "Baseline BERT (WordPiece)"))
all_results.append(train_model(bpe_tokenizer, "BERT + BPE"))
all_results.append(train_model(char_tokenizer, "BERT + Character-level"))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

In [ ]:
import pandas as pd
import json

df = pd.DataFrame([{
    "Model": r["name"],
    "Accuracy": f"{r['accuracy']:.2f}%",
    "F1 Score": f"{r['f1']:.2f}%",
    "Training Time": f"{r['training_time']/60:.1f} min",
    "Parameters": f"{r['parameters']/1e6:.1f}M"
} for r in all_results])

print(df.to_string(index=False))

with open("results.json", "w") as f:
    json.dump(all_results, f, indent=2)

                    Model Accuracy F1 Score Training Time Parameters
Baseline BERT (WordPiece)   26.60%   34.90%       4.0 min     108.9M
               BERT + BPE    2.60%    6.89%       4.2 min      94.2M
   BERT + Character-level    0.00%    0.00%       4.2 min      85.8M
